# GEOID-Flood — Colab entry point

Bootstrap only: this notebook performs setup and invokes the package CLI. Models and training are not implemented. Read `docs/COLAB.md`, `AGENTS.md`, contracts and approved ADRs. Select a GPU runtime for the Colab configuration. Do not download data or train without a separately approved task. Keep notebook outputs cleared.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/tungvoi38/final_cv.git"
COMMIT = "REPLACE_WITH_REVIEWED_40_CHARACTER_COMMIT"
CHECKOUT = Path("/content/final_cv")
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Select a reviewed immutable commit first"
assert (3, 11) <= sys.version_info[:2] < (3, 13), "Use a supported Python 3.11/3.12 runtime"
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(CHECKOUT)], check=True)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=CHECKOUT).strip(), (
    "Preserve local changes before checkout"
)
assert (
    subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CHECKOUT).decode().strip()
    == REPOSITORY
)
subprocess.run(["git", "fetch", "origin", COMMIT], cwd=CHECKOUT, check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=CHECKOUT, check=True)
assert (
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT).decode().strip() == COMMIT
)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.6"], check=True)
# Keep Colab's GPU-enabled torch visible; the core lock deliberately excludes torch.
if not (CHECKOUT / ".venv").exists():
    subprocess.run(
        ["uv", "venv", "--system-site-packages", "--python", sys.executable, ".venv"],
        cwd=CHECKOUT,
        check=True,
    )
subprocess.run(
    ["uv", "sync", "--python", sys.executable, "--locked", "--no-dev", "--no-editable"],
    cwd=CHECKOUT,
    check=True,
)

In [ ]:
os.environ["GEOID_DATA_ROOT"] = "/content/drive/MyDrive/geoid-flood/data/raw/geoid-flood"
os.environ["GEOID_CHECKPOINT_ROOT"] = "/content/drive/MyDrive/geoid-flood/checkpoints"
os.environ["GEOID_RUN_ROOT"] = "/content/drive/MyDrive/geoid-flood/runs"
EXPERIMENT = "configs/experiments/baseline.yaml"
ENVIRONMENT = "configs/environments/colab.yaml"
RESUME = None  # Set to a trusted full-state epoch-boundary checkpoint when implemented.
DRY_RUN = True  # Keep true during bootstrap; no model training is available.
command = [
    "uv",
    "run",
    "--python",
    sys.executable,
    "--locked",
    "--no-editable",
    "python",
    "scripts/train.py",
    "--config",
    EXPERIMENT,
    "--environment",
    ENVIRONMENT,
]
if RESUME:
    command += ["--resume", RESUME]
if DRY_RUN:
    command += ["--dry-run"]
subprocess.run(command, cwd=CHECKOUT, check=True)